In [1]:
# This script merges song features with staple scores and fills missing values. This will be used for playlist role annotation.
# Originally, we were going to use these for clustering analysis, but we found that the staple scores were too sparse and numerous, introducing too many dimensions.
import pandas as pd
import sqlite3

conn = sqlite3.connect('dbs/song_features.db')
song_features_df = pd.read_sql_query("SELECT * FROM song_features", conn)
conn.close()

staple_scores_df = pd.read_pickle('dbs/staple_scores.pkl')
staple_scores_df = staple_scores_df.reset_index().rename(columns={'index': 'track_uri'})

merged_df = pd.merge(
    song_features_df,
    staple_scores_df,
    how='left',
    on='track_uri'
)

theme_columns = staple_scores_df.columns.difference(['track_uri'])
merged_df[theme_columns] = merged_df[theme_columns].fillna(0)

In [7]:
# Role annotation
cluster_labels = {
    0: 'Generic',
    1: 'Niche',
    2: 'Mainstream',
    3: 'Curated'}

theme_names_list = staple_scores_df.columns.tolist()
staple_score_columns = [col for col in merged_df.columns if col in theme_names_list]
merged_df[staple_score_columns] = merged_df[staple_score_columns].apply(pd.to_numeric, errors='coerce')
merged_df['role_annotation'] = merged_df['cluster'].map(cluster_labels) + ' ' + merged_df[staple_score_columns].idxmax(axis=1).str.title() # cluster label + highest staple score theme

missing_theme = (merged_df[staple_score_columns].max(axis=1) == 0) | (merged_df[staple_score_columns].isna().all(axis=1)) # Identify rows with no staple scores
merged_df.loc[missing_theme, 'role_annotation'] = merged_df.loc[missing_theme, 'cluster'].map(cluster_labels) # Assign generic role (no theme label) for missing staple scores

In [ ]:
# Adding the annotated roles to song_features.db (if it exists, otherwise left blank)
# Note as reminder: Only the top 100 themes were discovered via TF-IDF due to the complexity and limitations of finding every theme,
# as this required a 1,000,000 x 1,000,000 matrix to be created, which is not feasible.

import sqlite3

conn = sqlite3.connect('dbs/song_features.db')
cursor = conn.cursor()
try:
    cursor.execute("ALTER TABLE song_features ADD COLUMN role_annotation TEXT")
    conn.commit()
    print("Added role_annotation column to song_features")
except sqlite3.OperationalError:
    print("Column role_annotation already exists in song_features, skipping addition.")

data_to_update = list(merged_df[['role_annotation', 'track_uri']].itertuples(index=False, name=None))

try: 
    cursor.executemany("UPDATE song_features SET role_annotation = ? WHERE track_uri = ?", data_to_update)
    conn.commit()
except sqlite3.Error as e:
    print(f"Error updating role_annotation in song_features, skipping update. Error: {e}")

conn.close()
print("Role annotation added to song_features successfully.")


Column role_annotation already exists in song_features, skipping addition.
